# 자습 12. 학습한 정책의 평가와 실패 분석

> **선택 자습 — 본수업 8시간에 포함하지 않습니다.** 코드 셀은 소형 합성 예제이며 GPU 학습·다운로드·로봇 구동을 자동 실행하지 않습니다. 실제 작업 명령은 설명의 터미널 예시로 제공하며 준비물을 먼저 갖춥니다. TODO는 직접 작성하고 아래 정답·해설로 검증하세요. [전체 자습 경로](selfstudy/README.md)


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| 평가 JSON 없음 / 성공률 null | 실행 경로와 완료/중단/미실행을 확인합니다. 분모 0을 0%로 바꾸지 않습니다. | 결과와 실행 기록 대응, 분모 명시 |
| 저장 입력 재생 성공, 폐루프 실패 | 카메라·단위·정규화·adapter·지연을 확인합니다. 재생 성공은 작업 성공이 아닙니다. | 같은 조건의 폐루프 결과 기록 |
| manifest/revision 검사 실패 | 본수업 고정 모델을 덮어쓰지 말고 새 정책 경로와 adapter를 사용합니다. | 모델·processor·계약의 버전 일치 |

**예시에 없는 오류를 확인할 곳:** [VLA 단계별 오류 사전](selfstudy/TROUBLESHOOTING.md)의 12 절과 [준비 안내](selfstudy/README.md). 훈련은 터미널 stdout/stderr와 checkpoint 설정, 재생은 지정 output의 결과 파일을 봅니다.

VLA 훈련·저장 관측 재생에는 noVNC 창이 열리지 않습니다. 본수업 GUI rollout을 다시 실행한 경우에만 FINISHED 후 MuJoCo 창을 닫습니다.

[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## loss가 내려가면 배포해도 되나요?

<!-- procedure:lesson-48-5c6070:start -->
![평가 범위를 단계적으로 넓히기](../교재그림/절차/lesson-48.svg)

대응 시뮬레이터가 없으면 LIBERO 결과를 자기 로봇의 성공으로 대체하지 않습니다.
<!-- procedure:lesson-48-5c6070:end -->


학습 loss는 시연에서의 objective 값입니다. 실제 환경에서는 정책 행동이 다음 관측을 바꾸므로 작은 오차가 누적될 수 있습니다. 따라서 **저장 관측 재생 → 시뮬레이션 폐루프 → 실제 장비의 제한된 시험** 순서로 평가 범위를 넓힙니다. 자신의 로봇과 대응되는 시뮬레이터가 없으면 LIBERO 성공을 그 로봇의 성공으로 대체하지 않습니다.

**완료 결과:** 기존 정책 대비 비교표, 실패 분류, 배포 보류/진행 판단의 근거. 소형 연습 60분, 실제 시험 시간은 별도입니다.

| 단계 | 확인하는 것 | 아직 말할 수 없는 것 |
|---|---|---|
| 저장 입력 재생 | shape·통계·출력 유한성·재로딩 일치 | 폐루프 작업 성공 |
| 시뮬레이션 rollout | 초기 조건별 성공·시간·종료 사유 | 실제 접촉·보정 오차에 대한 성능 |
| 제한된 실제 시험 | 장비 조건에서의 성공·중단·지연 | 보지 않은 모든 작업·장비에 대한 일반 성능 |

정책 A/B에 같은 초기 조건·작업 지시·실행 horizon·시간 제한·GUI 여부를 적용하고 가능한 한 같은 순서·조건으로 비교합니다. 학습에 사용하지 않은 날짜·물체·배치 등 group별 결과도 확인합니다. 총 성공률이 높아도 특정 조건에서 모두 실패할 수 있습니다.


## 새 checkpoint를 본수업 실행기에 그냥 넣으면 되나요?

아닙니다. `course_runtime.check_model`은 공개 LIBERO revision과 참고 manifest를 고정 검증합니다. 새로운 checkpoint를 그 경로에 덮어쓰거나 manifest를 위조해 통과시키지 않습니다. 자신의 모델은 별도 평가 어댑터와 경로를 사용합니다.

LIBERO를 계속 쓰는 경우에도 새 모델의 카메라 키, 상태8, 행동7, 정규화·후처리·controller 의미가 같아야 합니다. 원래 `make_env`를 재사용할 수 있지만 새 정책 loader·processor와 관측 adapter를 별도로 작성하고 저장 입력부터 검사합니다. 다른 로봇은 관측과 행동 adapter가 모두 달라집니다.

**평가 구현 TODO:** 로컬 checkpoint 읽기 → 그 checkpoint의 pre/postprocessor 읽기 → 같은 초기 조건 reset → 관측을 계약에 맞게 변환 → 행동 묶음 예측 → 앞 K개만 제어기에 적용 → 재관측을 구현합니다. 이 자료는 장비별 actuator 드라이버와 범용 실물 rollout 프로그램을 제공하지 않습니다. 배포 입력 재생 명령은 13에서 제공합니다.


## 무엇을 로그에 남기나요?

실행 ID, checkpoint hash, 데이터/코드 버전, task·초기 조건, success, 종료 사유, 적용 step 수, 전체 시간, 추론 지연 분포, 관측 나이, 오류·중단 이유를 남깁니다. 시간이 초과되면 완료 실패인지 실행 중단인지 기준을 실험 전에 정합니다. 운영 관점에서는 전체 시도 대비 성공도 함께 보고해 중단이 많은 정책의 완료 성공률만 좋아 보이는 문제를 막습니다.

성공 여부는 사람의 영상 판정이라면 기준과 평가자를 기록하고, 환경 센서 기준이라면 신호와 조건을 기록합니다. 평균 추론 시간만으로 제어 주기를 정하지 말고 느린 꼬리 구간과 capture→추론→적용 전체 지연도 봅니다. 07처럼 추론 중 물리 환경을 멈추는 모드는 실제 장비에서 센서와 물체가 계속 움직이는 상황과 다릅니다.


## TODO — 완료 실패와 실행 중단을 구분하기

아래 합성 4회는 완료 성공 2, 완료 실패 1, 중단 1입니다. 완료 기준 성공률과 전체 시도 기준 성공률을 각각 반환하세요. 완료 기록이 하나도 없으면 완료 기준 성공률은 `None`입니다.


In [ ]:
runs = [
    {'status':'completed', 'success':True},
    {'status':'completed', 'success':False},
    {'status':'incomplete', 'success':False},
    {'status':'completed', 'success':True},
]
def score_runs(rows):
    # TODO: completed_rate, attempt_rate, incomplete 키를 가진 dict
    return None
report = score_runs(runs)
if report is None:
    print('TODO 미완료: 평가 분모를 구분하세요.')
else:
    assert abs(report['completed_rate']-2/3)<1e-9
    assert report['attempt_rate']==0.5 and report['incomplete']==1
    assert score_runs([{'status':'incomplete','success':False}])['completed_rate'] is None
    print('합성 평가:', report)

### 정답과 해설

```python
def score_runs(rows):
    completed = [r for r in rows if r['status']=='completed']
    successes = sum(bool(r['success']) for r in completed)
    return {
        'completed_rate': successes/len(completed) if completed else None,
        'attempt_rate': successes/len(rows) if rows else None,
        'incomplete': len(rows)-len(completed),
    }
```

완료 기준은 2/3, 전체 시도 기준은 2/4입니다. 두 값은 서로 모순되지 않고 질문이 다릅니다. 아직 실행하지 않은 조건은 rows에 실패로 추가하지 않고 missing으로 별도 보고합니다. 소수 표본의 비율만으로 일반 성공률을 보장하지 않으며 조건·횟수·불확실성을 함께 기록합니다.

**추가 TODO:** 기준 정책은 8/10 완료 성공·중단0, 후보 정책은 8/8 완료 성공·중단2였다면 어떤 정책을 선택할까요?

**해설:** 전체 시도 성공은 둘 다 8/10입니다. 후보의 완료 성공률 100%만으로 개선이라고 말할 수 없습니다. 중단 원인·안전성·지연·동일 조건 재현을 확인한 후 결정합니다.


## 실패를 다음 작업으로 연결하기

| 관찰 | 먼저 조사할 것 | 다음 작업 |
|---|---|---|
| 시작부터 반대 방향 이동 | 관절 순서·단위·좌표계·카메라 방향 | adapter 검사와 알려진 입력 재생 |
| 잡기 직전 반복 실패 | 시연의 접촉 구간·카메라 가림 | 해당 구간의 시연·관측 개선 |
| 처음은 맞고 점점 이탈 | 분포 이동·horizon·지연 | 짧은 horizon 비교, 복구 시연 수집 |
| loss만 좋아지고 시험 실패 | 데이터 누수·과적합·조건 편중 | 그룹 분할 재검토, validation 비교 |
| 입력 끊김 후 계속 움직임 | 오래된 명령·watchdog 누락 | 제어기 중단 경로와 13의 명령 수명 검사 |

정답 해설은 원인을 하나로 단정하지 않습니다. 한 번에 한 가설을 바꾸어 비교하고, 새 데이터로 학습한 경우 dataset revision과 checkpoint hash를 바꿉니다. 실패 영상을 삭제해 성공 사례만 남기지 않습니다.

다음: [13 로컬 배포](13_selfstudy_local_deployment.ipynb). 본수업의 평가와 상태 분리는 [08](08_evaluation.ipynb), 구현 기준은 [evaluate.py](scripts/evaluate.py)에서 확인합니다.
